# Features et preprocessing

Ce notebook documente les features actives, pourquoi elles le sont, et ce qui a été testé puis écarté.

## Retrait de la standardisation et variables catégorielles natives

LightGBM (modèle à base d'arbres) est invariant à toute transformation monotone des features numériques. Standardiser n'y change aucune décision de split, ça ajoute juste un artefact à maintenir. Le one-hot sur les catégorielles explose la dimensionnalité pour un gain nul.

Cependant, les codes catégoriels doivent rester stables entre entraînement et inférence, sinon un lot contenant un sous-ensemble des catégories vues à l'entraînement (une machine "H" appelée seule, par exemple) serait mal interprété.

## Pourquoi OSF reste le point faible

Trois hypothèses plausibles ont été testées pour améliorer la détection OSF au-delà du signal court (24h) et bruité déjà identifié. Mais, les trois ont échoué :

1. **Classifieur OSF dédié** (au lieu du modèle mutualisé) : plafonne à 43,3 % de recall, pire que le modèle mutualisé (53,3 %). Seulement 23 pannes OSF en train, pas assez pour qu'un classifieur isolé apprenne quoi que ce soit de robuste.
2. **Horizons différenciés par mode** (72h pour HDF/OSF au lieu de 120h uniforme) : réduire la fenêtre réduit aussi le nombre d'exemples positifs disponibles, ce qui coûte plus cher que le bruit retiré n'en fait gagner.
3. **Sur-pondération des exemples OSF** (x2 à x5) : neutre au mieux, dégradé au pire.

## Passage à 100 machines

OSF s'améliore légèrement (+1,8 à +5,4 pts pour les trois méthodes testées) mais **HDF se dégrade nettement, y compris pour la règle triviale qui n'apprend rien**. C'est le signe d'un effet de composition d'échantillon (nouvelle population de machines), pas d'architecture. Un seul run à seeds différents ne permet pas de distinguer un vrai effet d'un artefact d'échantillonnage. Le dataset reste à 50 machines.

## Importances de features

In [1]:
import sys
sys.path.append('..')

import joblib
import pandas as pd

calibreur = joblib.load("../models/model_pipeline.pkl")
pipeline_entraine = calibreur.calibrated_classifiers_[0].estimator.estimator
classifieur = pipeline_entraine.named_steps["classifier"]

noms = [n.replace("num__", "").replace("cat__", "") for n in classifieur.booster_.feature_name()]
importances = pd.Series(classifieur.feature_importances_, index=noms).sort_values(ascending=False)
print(importances.head(10))


usure_outil                        448
encrassement_max_24h               231
encrassement                       214
vibration_max_24h                  179
poussiere_max_24h                  128
ecart_thermique_max_24h            118
temperature_process_max_24h        115
couple_max_24h                     104
temperature_process_moyenne_24h     97
type_machine                        93
dtype: int32
